# Project Gutenberg Author Downloader: Scrape, Parse, and Save Texts by Author

In [27]:
"""
Notebook: Gutenberg Author Scraper & Text Exporter

This notebook downloads all available books for a given author ID from Project Gutenberg.
It extracts metadata (title, author, download link), fetches plain text versions, and
saves each book into a folder structure like:

    /mnt/mls/data/books/
        ├── edgar_allan_poe/
        │   ├── the_raven.txt
        │   ├── the_masque_of_the_red_death.txt
        │   └── ...
        ├── unknown/
        │   └── various_untagged_titles.txt

Dependencies: requests, parsel, pathlib, re, pandas

To use:
- Update `author_id` to target another author (from Gutenberg).
- Ensure export directory `/mnt/mls/data/books/` is writable and accessible.
"""

import pandas as pd
import re
import requests
from parsel import Selector
from pathlib import Path
import os


import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

print("Notebook sees this path:", os.getcwd())


Notebook sees this path: /mnt


In [28]:
# Author and source settings
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}
base_uri = 'https://www.gutenberg.org'
author_id = 481  # Edgar Allan Poe

# Scrape book listing page
resp = requests.get(f'{base_uri}/ebooks/author/{author_id}', headers=headers, timeout=5, verify=False)
sel = Selector(text=resp.text)


In [29]:
# Parse books from the author page
books = sel.xpath('//li[@class="booklink"]')
books_info_list = []

for book in books:
    book_info_dict = dict()
    book_info_dict['author_id'] = author_id
    book_info_dict['author'] = book.xpath('.//span[contains(@class,"subtitle")]/text()').extract_first()
    book_info_dict['title'] = book.xpath('.//span[contains(@class,"title")]/text()').extract_first()
    book_info_dict['href'] = book.xpath('.//a/@href').extract_first()

    result = re.search(r'/ebooks/(\d+)', book_info_dict['href'])
    if result:
        book_info_dict['book_id'] = result.group(1)
        book_info_dict['filename'] = f"pg{book_info_dict['book_id']}.txt"
        book_info_dict['download_url'] = f"{base_uri}/cache/epub/{book_info_dict['book_id']}/{book_info_dict['filename']}"
        books_info_list.append(book_info_dict)


In [30]:
# Preview parsed book info
pd.DataFrame(books_info_list).head()


,author_id,author,title,href,book_id,filename,download_url
0,481,Edgar Allan Poe,The Works of Edgar Allan Poe — Volume 2,/ebooks/2148,2148,pg2148.txt,https://www.gutenberg.org/cache/epub/2148/pg21...
1,481,Edgar Allan Poe,The Works of Edgar Allan Poe — Volume 1,/ebooks/2147,2147,pg2147.txt,https://www.gutenberg.org/cache/epub/2147/pg21...
2,481,Edgar Allan Poe,"The Works of Edgar Allan Poe, The Raven Edition",/ebooks/25525,25525,pg25525.txt,https://www.gutenberg.org/cache/epub/25525/pg2...
3,481,Edgar Allan Poe,The Fall of the House of Usher,/ebooks/932,932,pg932.txt,https://www.gutenberg.org/cache/epub/932/pg932...
4,481,Edgar Allan Poe,The Raven,/ebooks/17192,17192,pg17192.txt,https://www.gutenberg.org/cache/epub/17192/pg1...


In [31]:
# Helper function to retrieve book text
def return_book_text(book_info: dict) -> str:
    resp = requests.get(book_info['download_url'], headers=headers, verify=False)
    return resp.text


In [32]:
# Directory for storing all books
books_export_dir = Path('/mnt/mls/data/books')
books_export_dir.mkdir(parents=True, exist_ok=True)

# Save each book to its author's folder
for book_info in books_info_list:
    title = book_info.get('title', "untitled")
    author = book_info.get('author') or "unknown"

    # Clean names for filesystem
    safe_title = re.sub(r'[\W_]+', '_', title.lower()).strip('_')
    safe_author = re.sub(r'[\W_]+', '_', author.lower()).strip('_')

    # Create export path per author
    export_path = books_export_dir / safe_author
    export_path.mkdir(parents=True, exist_ok=True)

    # Build and write to file
    filename = f"{safe_title}.txt"
    full_path = export_path / filename
    print(f'Saving to {full_path}')

    try:
        with open(full_path, 'w', encoding='utf-8') as f:
            f.write(return_book_text(book_info))
    except Exception as e:
        print(f"⚠️ Failed to save '{title}': {e}")


Saving to /mnt/mls/data/books/edgar_allan_poe/the_works_of_edgar_allan_poe_volume_2.txt
Saving to /mnt/mls/data/books/edgar_allan_poe/the_works_of_edgar_allan_poe_volume_1.txt
Saving to /mnt/mls/data/books/edgar_allan_poe/the_works_of_edgar_allan_poe_the_raven_edition.txt
Saving to /mnt/mls/data/books/edgar_allan_poe/the_fall_of_the_house_of_usher.txt
Saving to /mnt/mls/data/books/edgar_allan_poe/the_raven.txt
Saving to /mnt/mls/data/books/edgar_allan_poe/the_masque_of_the_red_death.txt
Saving to /mnt/mls/data/books/edgar_allan_poe/the_cask_of_amontillado.txt
Saving to /mnt/mls/data/books/edgar_allan_poe/the_narrative_of_arthur_gordon_pym_of_nantucket.txt
Saving to /mnt/mls/data/books/unknown/the_best_american_humorous_short_stories.txt
Saving to /mnt/mls/data/books/edgar_allan_poe/the_complete_poetical_works_of_edgar_allan_poe.txt
Saving to /mnt/mls/data/books/unknown/a_collection_of_short_stories.txt
Saving to /mnt/mls/data/books/edgar_allan_poe/the_works_of_edgar_allan_poe_volume_5.